In [ ]:
import pandas as pd
import numpy as np

print("Avvio Modulo 01: Data Quality & Integrity Check...\n")

# --- 1. CONFIGURAZIONE PERCORSI ASSOLUTI (RAW STRINGS) ---
file_fut_03 = r"C:\Users\Leona\source\repos\QuantEdge_Project\data\raw\ES 03-26.Last.txt"
file_fut_06 = r"C:\Users\Leona\source\repos\QuantEdge_Project\data\raw\ES 06-26.Last.txt"
file_cfd = r"C:\Users\Leona\source\repos\QuantEdge_Project\data\raw\fpmarkets_us500_ticks.csv"

# --- 2. FUNZIONE DI SCANSIONE ULTRA-LEGGERA FUTURE ---
def analizza_integrita_future(file_path):
    print(f"Scansione integrità Future: {file_path}")
    
    # Leggiamo solo la colonna 0 (Timestamp) per non intaccare la memoria
    df = pd.read_csv(file_path, sep=';', usecols=[0], names=['RawTime'], engine='c')
    
    # Pulizia e conversione in formato data
    df['CleanTime'] = df['RawTime'].str.slice(0, 15)
    df['Datetime'] = pd.to_datetime(df['CleanTime'], format='%Y%m%d %H%M%S')
    
    # Calcolo della derivata temporale: quanto tempo passa tra una riga e la successiva?
    df['Time_Gap'] = df['Datetime'].diff()
    
    # Isoliamo i "buchi" anomali. 
    # Fisiologicamente, il CME chiude 1 ora al giorno e 48 ore nel weekend.
    # Segnaliamo qualsiasi interruzione superiore alle 3 ore.
    soglia_allarme = pd.Timedelta(hours=3)
    gaps_anomali = df[df['Time_Gap'] > soglia_allarme]
    
    print(f"  -> Righe rilevate: {len(df):,}")
    print(f"  -> Primo Tick: {df['Datetime'].min()}")
    print(f"  -> Ultimo Tick: {df['Datetime'].max()}")
    print(f"  -> Pause di contrattazione o Weekend rilevati: {len(gaps_anomali)}")
    print("-" * 50)
    
    # Liberiamo la RAM forzatamente
    del df
    return gaps_anomali

# --- 3. FUNZIONE DI SCANSIONE CFD (CON FIX ISO8601) ---
def analizza_integrita_cfd(file_path):
    print(f"Scansione integrità CFD: {file_path}")
    
    # Leggiamo solo la colonna del tempo
    df = pd.read_csv(file_path, sep=';', usecols=['datetime_utc'], engine='c')
    
    # FIX: format='ISO8601' gestisce sia i millisecondi presenti che quelli troncati
    df['Datetime'] = pd.to_datetime(df['datetime_utc'], format='ISO8601')
    
    print(f"  -> Righe rilevate: {len(df):,}")
    print(f"  -> Primo Tick: {df['Datetime'].min()}")
    print(f"  -> Ultimo Tick: {df['Datetime'].max()}")
    print("-" * 50)
    
    del df

# --- 4. ESECUZIONE DIAGNOSTICA ---
gaps_03 = analizza_integrita_future(file_fut_03)
gaps_06 = analizza_integrita_future(file_fut_06)
analizza_integrita_cfd(file_cfd)

print("\nDIAGNOSTICA COMPLETATA. Dati pronti per il modulo 02 (Merge Vettoriale).")

Avvio Modulo 01: Data Quality & Integrity Check...

Scansione integrità Future: C:\Users\Leona\source\repos\QuantEdge_Project\data\raw\ES 03-26.Last.txt
  -> Righe rilevate: 58,024,732
  -> Primo Tick: 2025-12-28 16:38:09
  -> Ultimo Tick: 2026-03-12 22:59:55
  -> Pause di contrattazione o Weekend rilevati: 20
--------------------------------------------------
Scansione integrità Future: C:\Users\Leona\source\repos\QuantEdge_Project\data\raw\ES 06-26.Last.txt
  -> Righe rilevate: 69,184,613
  -> Primo Tick: 2026-03-12 23:00:00
  -> Ultimo Tick: 2026-06-10 14:44:09
  -> Pause di contrattazione o Weekend rilevati: 14
--------------------------------------------------
Scansione integrità CFD: C:\Users\Leona\source\repos\QuantEdge_Project\data\raw\fpmarkets_us500_ticks.csv


ValueError: time data "2026-02-09 01:00:11+00:00" doesn't match format "%Y-%m-%d %H:%M:%S.%f%z", at position 40. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.